# LeetCode 1642 — Furthest Building You Can Reach bằng Python

Link đề: https://leetcode.com/problems/furthest-building-you-can-reach/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def furthestBuilding(self, heights: List[int], bricks: int, ladders: int) -> int:
        ...
```

Notebook đi từ **thử mọi cách dùng gạch/thang** (vét cạn) → **quy hoạch động** theo số thang đã dùng → **tìm kiếm nhị phân đáp án** → hai cách **tham lam với heap**: min-heap kích thước `ladders` (lời giải chính) và max-heap "đổi gạch lấy thang".

## 1. Mục tiêu học

1. Mô hình hóa bài: chỉ các bước **leo lên** (`d = h[i+1] − h[i] > 0`) mới tốn tài nguyên; mỗi bước leo trả bằng `d` viên gạch **hoặc** 1 cái thang.
2. Hiểu nhận xét tham lam then chốt: với một tập bước leo cố định, nên dùng **thang cho `ladders` bước cao nhất**, gạch cho phần còn lại.
3. Thành thạo `heapq` (min-heap) và mẹo **đổi dấu** để có max-heap trong Python.
4. Biết ba lối đi tới cùng đáp án: DP `O(n·L)`, tìm kiếm nhị phân đáp án `O(n log² n)`, heap `O(n log L)`.
5. Viết test đối chiếu với brute force, chú ý lỗi **lệch một** khi trả về chỉ số tòa nhà.

## 2. Đề bài & ví dụ

Cho mảng `heights` (chiều cao các tòa nhà), số **gạch** `bricks` và số **thang** `ladders`. Bạn xuất phát ở tòa `0` và đi lần lượt sang tòa kế tiếp. Khi đi từ tòa `i` sang tòa `i + 1`:
- Nếu `heights[i] ≥ heights[i+1]`: đi xuống/ngang, **không** tốn gì.
- Nếu `heights[i] < heights[i+1]`: phải dùng **1 thang** hoặc **`heights[i+1] − heights[i]` viên gạch**.

Trả về **chỉ số** (từ 0) của tòa xa nhất có thể tới nếu dùng gạch và thang tối ưu.

**Ví dụ 1:** `heights = [4,2,7,6,9,14,12]`, `bricks = 5`, `ladders = 1` → `4`.
- 0→1: `4 ≥ 2`, miễn phí. 1→2: leo 5, dùng 5 gạch. 2→3: miễn phí. 3→4: leo 3, dùng thang. 4→5 cần leo 5 nhưng hết tài nguyên.

**Ví dụ 2:** `heights = [4,12,2,7,3,18,20,3,19]`, `bricks = 10`, `ladders = 2` → `7`.

**Ví dụ 3:** `heights = [14,3,19,3]`, `bricks = 17`, `ladders = 0` → `3`.

**Ràng buộc:**
- `1 ≤ heights.length ≤ 10⁵`
- `1 ≤ heights[i] ≤ 10⁶`
- `0 ≤ bricks ≤ 10⁹`
- `0 ≤ ladders ≤ heights.length`

In [1]:
import heapq
import random
import time
from typing import List
from functools import lru_cache

examples = [
    ([4, 2, 7, 6, 9, 14, 12], 5, 1, 4),
    ([4, 12, 2, 7, 3, 18, 20, 3, 19], 10, 2, 7),
    ([14, 3, 19, 3], 17, 0, 3),
]

def climbs(heights):
    """Danh sách (chỉ số bước i, độ cao phải leo d) cho các bước đi lên."""
    return [(i, heights[i + 1] - heights[i]) for i in range(len(heights) - 1) if heights[i + 1] > heights[i]]

for h, b, l, exp in examples:
    print(f"heights={h}, bricks={b}, ladders={l} -> kỳ vọng {exp}")
    print("   các bước leo (i -> i+1, d):", climbs(h))

heights=[4, 2, 7, 6, 9, 14, 12], bricks=5, ladders=1 -> kỳ vọng 4
   các bước leo (i -> i+1, d): [(1, 5), (3, 3), (4, 5)]
heights=[4, 12, 2, 7, 3, 18, 20, 3, 19], bricks=10, ladders=2 -> kỳ vọng 7
   các bước leo (i -> i+1, d): [(0, 8), (2, 5), (4, 15), (5, 2), (7, 16)]
heights=[14, 3, 19, 3], bricks=17, ladders=0 -> kỳ vọng 3
   các bước leo (i -> i+1, d): [(1, 16)]


## 3. Kiến thức nền

### 3.1 Hiệu hai phần tử liên tiếp

Ta chỉ quan tâm `d = heights[i+1] − heights[i]`. `zip(heights, heights[1:])` cho từng cặp liên tiếp; `max(0, d)` biến bước đi xuống thành chi phí 0.

In [2]:
h = [4, 2, 7, 6, 9, 14, 12]
diffs = [b - a for a, b in zip(h, h[1:])]
print("hiệu:", diffs)
print("chi phí bằng gạch:", [max(0, d) for d in diffs])

hiệu: [-2, 5, -1, 3, 5, -2]
chi phí bằng gạch: [0, 5, 0, 3, 5, 0]


### 3.2 `heapq` — min-heap

`heapq` biến một list thành **min-heap**: `heap[0]` luôn là phần tử **nhỏ nhất**. `heappush` / `heappop` tốn `O(log k)` với `k` là kích thước heap. `heappushpop(heap, x)` đẩy `x` rồi lấy ra phần tử nhỏ nhất — nhanh hơn gọi hai lệnh.

In [3]:
heap = []
for x in [5, 3, 8, 1]:
    heapq.heappush(heap, x)
    print(f"push {x}: heap={heap}, nhỏ nhất={heap[0]}")
print("pop ->", heapq.heappop(heap), "| còn", heap)
print("heappushpop(heap, 2) ->", heapq.heappushpop(heap, 2), "| còn", heap)

push 5: heap=[5], nhỏ nhất=5
push 3: heap=[3, 5], nhỏ nhất=3
push 8: heap=[3, 5, 8], nhỏ nhất=3
push 1: heap=[1, 3, 8, 5], nhỏ nhất=1
pop -> 1 | còn [3, 5, 8]
heappushpop(heap, 2) -> 2 | còn [3, 5, 8]


### 3.3 Max-heap bằng cách đổi dấu

Python không có max-heap sẵn. Mẹo: đẩy `-x` vào min-heap; phần tử "nhỏ nhất" là `-max`.

In [4]:
mx = []
for x in [5, 3, 8, 1]:
    heapq.heappush(mx, -x)
print("lớn nhất =", -mx[0])
print("lấy ra lần lượt:", [-heapq.heappop(mx) for _ in range(4)])

lớn nhất = 8
lấy ra lần lượt: [8, 5, 3, 1]


### 3.4 Tìm kiếm nhị phân trên đáp án

Nếu một tính chất `ok(k)` **đơn điệu** (đúng với `k` thì đúng với mọi `k' < k`), ta tìm `k` lớn nhất thỏa mãn bằng tìm kiếm nhị phân: `O(log n)` lần gọi `ok`.

In [5]:
def last_true(lo, hi, ok):
    """k lớn nhất trong [lo, hi] với ok(k) đúng; giả sử ok(lo) đúng và ok đơn điệu."""
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print(last_true(0, 100, lambda k: k * k <= 50))

7


## 4. Ý tưởng nền tảng / trực giác

Xét ví dụ 1: các bước leo là `d = 5` (1→2), `3` (3→4), `5` (4→5). Có 5 gạch, 1 thang.

Hai cách cho hai bước leo đầu:
- **Thang cho 5, gạch cho 3:** còn 2 gạch. Bước 4→5 cần 5 ⇒ dừng ở tòa 4.
- **Gạch cho 5, thang cho 3:** còn 0 gạch. Cũng dừng ở tòa 4.

Kết quả bằng nhau, nhưng ý tưởng quan trọng lộ ra: **thang "đáng giá" nhất khi dùng cho bước leo cao nhất** — một thang thay được bao nhiêu gạch cũng được. Nếu muốn đi qua một tập bước leo `D`, cách tốt nhất là dùng thang cho `ladders` phần tử **lớn nhất** của `D`, phần còn lại trả bằng gạch. Đi được ⇔ `tổng(D) − tổng(L lớn nhất của D) ≤ bricks`.

Tính tay ví dụ 2: `heights = [4,12,2,7,3,18,20,3,19]`, 10 gạch, 2 thang. Các bước leo theo thứ tự: `8` (0→1), `5` (2→3), `15` (4→5), `2` (5→6), `16` (7→8).
- Đến tòa 6: `D = {8, 5, 15, 2}`, thang cho `15, 8`, gạch `5 + 2 = 7 ≤ 10` ✔.
- Đến tòa 7: bước 6→7 đi xuống, vẫn ✔.
- Đến tòa 8: thêm `16`: thang cho `16, 15`, gạch `8 + 5 + 2 = 15 > 10` ✘ ⇒ đáp án `7`.

In [6]:
def can_reach_by_rule(heights, bricks, ladders, k):
    """Có tới được tòa k không? Áp dụng quy tắc 'thang cho L bước cao nhất'."""
    D = sorted((max(0, heights[i + 1] - heights[i]) for i in range(k)), reverse=True)
    return sum(D[ladders:]) <= bricks

h, b, l = [4, 12, 2, 7, 3, 18, 20, 3, 19], 10, 2
for k in range(len(h)):
    D = sorted((max(0, h[i + 1] - h[i]) for i in range(k)), reverse=True)
    D = [d for d in D if d > 0]
    print(f"tới tòa {k}: bước leo {D}, thang cho {D[:l]}, gạch cần {sum(D[l:])} -> {'✔' if can_reach_by_rule(h, b, l, k) else '✘'}")

tới tòa 0: bước leo [], thang cho [], gạch cần 0 -> ✔
tới tòa 1: bước leo [8], thang cho [8], gạch cần 0 -> ✔
tới tòa 2: bước leo [8], thang cho [8], gạch cần 0 -> ✔
tới tòa 3: bước leo [8, 5], thang cho [8, 5], gạch cần 0 -> ✔
tới tòa 4: bước leo [8, 5], thang cho [8, 5], gạch cần 0 -> ✔
tới tòa 5: bước leo [15, 8, 5], thang cho [15, 8], gạch cần 5 -> ✔
tới tòa 6: bước leo [15, 8, 5, 2], thang cho [15, 8], gạch cần 7 -> ✔
tới tòa 7: bước leo [15, 8, 5, 2], thang cho [15, 8], gạch cần 7 -> ✔
tới tòa 8: bước leo [16, 15, 8, 5, 2], thang cho [16, 15], gạch cần 15 -> ✘


## 5. Cách 1 — Brute Force / baseline: thử cả hai lựa chọn ở mỗi bước leo

**Ý tưởng:** đệ quy `go(i, b, l)` = tòa xa nhất tới được khi đang ở tòa `i`, còn `b` gạch và `l` thang. Ở mỗi bước leo, thử **cả** dùng gạch (nếu đủ) **và** dùng thang (nếu còn), lấy kết quả lớn hơn.

In [7]:
def furthest_brute(heights, bricks, ladders):
    n = len(heights)
    def go(i, b, l):
        if i == n - 1:
            return i
        d = heights[i + 1] - heights[i]
        if d <= 0:
            return go(i + 1, b, l)
        best = i                       # đứng lại ở i nếu không đi tiếp được
        if b >= d:
            best = max(best, go(i + 1, b - d, l))
        if l > 0:
            best = max(best, go(i + 1, b, l - 1))
        return best
    return go(0, bricks, ladders)

for h, b, l, exp in examples:
    print(furthest_brute(h, b, l), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
7 (kỳ vọng 7 )
3 (kỳ vọng 3 )


**Độ phức tạp:** mỗi bước leo phân nhánh 2 ⇒ `O(2^c)` với `c` là số bước leo (tới `10⁵`). Độ sâu đệ quy tới `n` — cũng vượt giới hạn đệ quy của Python khi `n` lớn. Chỉ dùng để kiểm thử với `n` nhỏ.

**Nút thắt:** ta thử lại rất nhiều trạng thái `(i, b, l)` tương đương; và quan trọng hơn, **ta chưa dùng** nhận xét "thang cho bước cao nhất".

## 6. Ý tưởng thuật toán / bước đột phá

**Nhận xét hoán đổi (exchange argument):** giả sử trong một phương án, có bước leo `x` dùng **thang** và bước leo `y > x` dùng **gạch**. Đổi lại (thang cho `y`, gạch cho `x`) thì số gạch dùng giảm `y − x > 0`, số thang không đổi ⇒ phương án mới **không tệ hơn**. Lặp lại ⇒ tồn tại phương án tối ưu mà thang luôn đi với các bước **cao nhất**.

**Hệ quả:** "tới được tòa `k`" ⇔ `tổng các bước leo trong [0, k) − tổng L bước leo lớn nhất trong đó ≤ bricks`. Khi `k` tăng, vế trái **không giảm** ⇒ tính chất **đơn điệu** ⇒ có thể tìm kiếm nhị phân, hoặc tốt hơn: quét từ trái sang phải và **duy trì** tập `L` bước leo lớn nhất bằng heap.

**Quy tắc khi quét (min-heap):**
- Gặp bước leo `d`: đẩy `d` vào min-heap (tạm "dùng thang" cho nó).
- **Nếu** heap có hơn `ladders` phần tử **thì** bước nhỏ nhất trong heap không xứng đáng dùng thang ⇒ lấy ra, trả bằng gạch: `bricks −= nhỏ nhất`.
- **Nếu** `bricks < 0` **thì** không thể bước sang tòa `i + 1` ⇒ trả về `i`.

**Bất biến:** sau khi xử lý bước `i → i+1`, heap chứa đúng `min(L, số bước leo)` bước leo **lớn nhất** đã gặp; `bricks` đã trừ tổng các bước leo còn lại.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quy hoạch động theo số thang đã dùng

`dp[j]` = số gạch **ít nhất** đã tiêu để tới tòa hiện tại khi đã dùng đúng `j` thang (`0 ≤ j ≤ L`). Gặp bước leo `d`:
`new[j] = min(dp[j] + d, dp[j − 1])` (gạch cho bước này, hoặc thang). Nếu `min(new) > bricks` thì không bước tiếp được.

In [8]:
def furthest_dp(heights, bricks, ladders):
    INF = float('inf')
    L = min(ladders, len(heights))
    dp = [0] + [INF] * L                 # chưa leo gì: 0 gạch, 0 thang
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d <= 0:
            continue
        new = [INF] * (L + 1)
        for j in range(L + 1):
            if dp[j] + d <= bricks:
                new[j] = dp[j] + d
            if j > 0 and dp[j - 1] < new[j]:
                new[j] = dp[j - 1]
        if min(new) > bricks:
            return i
        dp = new
    return len(heights) - 1

for h, b, l, exp in examples:
    print(furthest_dp(h, b, l), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
7 (kỳ vọng 7 )
3 (kỳ vọng 3 )


**Độ phức tạp:** `O(n · L)` thời gian, `O(L)` bộ nhớ. Đúng nhưng với `n = L = 10⁵` là `10¹⁰` — quá chậm. DP này **không dùng** nhận xét tham lam; nó cho thấy nhận xét đó tiết kiệm được bao nhiêu.

### 7.2 Cách 3 — Tìm kiếm nhị phân đáp án

`ok(k)` = có tới được tòa `k`: lấy các bước leo trong `[0, k)`, sắp giảm dần, tổng phần sau `L` phần tử đầu `≤ bricks`. `ok` đơn điệu và `ok(0)` luôn đúng ⇒ tìm `k` lớn nhất.

In [9]:
def furthest_binary(heights, bricks, ladders):
    n = len(heights)
    def ok(k):
        D = sorted((heights[i + 1] - heights[i] for i in range(k) if heights[i + 1] > heights[i]), reverse=True)
        return sum(D[ladders:]) <= bricks
    lo, hi = 0, n - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

for h, b, l, exp in examples:
    print(furthest_binary(h, b, l), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
7 (kỳ vọng 7 )
3 (kỳ vọng 3 )


In [10]:
class SolutionBinary:
    def furthestBuilding(self, heights: List[int], bricks: int, ladders: int) -> int:
        def ok(k: int) -> bool:
            D = sorted((heights[i + 1] - heights[i] for i in range(k) if heights[i + 1] > heights[i]), reverse=True)
            return sum(D[ladders:]) <= bricks
        lo, hi = 0, len(heights) - 1
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

print([SolutionBinary().furthestBuilding(h, b, l) for h, b, l, _ in examples])

[4, 7, 3]


**Độ phức tạp:** `O(log n)` lần gọi `ok`, mỗi lần sắp xếp `O(n log n)` ⇒ `O(n log² n)` thời gian, `O(n)` bộ nhớ. Nộp được (`10⁵` phần tử), nhưng chậm hơn heap.

### 7.3 Cách 4 — Min-heap kích thước `ladders` (lời giải chính)

Từng đoạn:
1. Duyệt `i` từ `0` tới `n − 2`, tính `d`.
2. Nếu `d > 0`: `heappush(heap, d)`.
3. Nếu `len(heap) > ladders`: bước nhỏ nhất bị "giáng cấp" xuống dùng gạch: `bricks −= heappop(heap)`.
4. Nếu `bricks < 0`: trả `i` (không sang được `i + 1`).
5. Hết vòng: tới được tòa cuối `n − 1`.

In [11]:
def furthest_minheap(heights, bricks, ladders):
    heap = []
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d <= 0:
            continue
        heapq.heappush(heap, d)
        if len(heap) > ladders:
            bricks -= heapq.heappop(heap)
            if bricks < 0:
                return i
    return len(heights) - 1

for h, b, l, exp in examples:
    print(furthest_minheap(h, b, l), "(kỳ vọng", exp, ")")

4 (kỳ vọng 4 )
7 (kỳ vọng 7 )
3 (kỳ vọng 3 )


**Độ phức tạp:** mỗi bước leo một `push` và nhiều nhất một `pop` trên heap kích thước `≤ L + 1` ⇒ `O(n log L)` thời gian, `O(L)` bộ nhớ.

### 7.4 Cách 5 — Max-heap: "dùng gạch trước, hết gạch thì đổi lấy thang"

Cách nghĩ ngược lại: luôn trả bằng gạch, đẩy `d` vào **max-heap**. Khi `bricks < 0`, nếu còn thang thì "hồi tố" dùng thang cho bước leo **lớn nhất** đã trả bằng gạch: lấy ra `big` từ max-heap, `bricks += big`, `ladders −= 1`. Nếu hết thang mà vẫn `bricks < 0` ⇒ trả `i`.

In [12]:
def furthest_maxheap(heights, bricks, ladders):
    heap = []                                  # max-heap bằng số âm
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d <= 0:
            continue
        bricks -= d
        heapq.heappush(heap, -d)
        if bricks < 0:
            if ladders == 0:
                return i
            bricks += -heapq.heappop(heap)     # dùng thang cho bước lớn nhất từng trả bằng gạch
            ladders -= 1
    return len(heights) - 1

class SolutionMaxHeap:
    def furthestBuilding(self, heights: List[int], bricks: int, ladders: int) -> int:
        heap = []
        for i in range(len(heights) - 1):
            d = heights[i + 1] - heights[i]
            if d > 0:
                bricks -= d
                heapq.heappush(heap, -d)
                if bricks < 0:
                    if ladders == 0:
                        return i
                    bricks -= heapq.heappop(heap)   # pop trả về -big
                    ladders -= 1
        return len(heights) - 1

for h, b, l, exp in examples:
    print(furthest_maxheap(h, b, l), SolutionMaxHeap().furthestBuilding(h, b, l), "(kỳ vọng", exp, ")")

4 4 (kỳ vọng 4 )
7 7 (kỳ vọng 7 )
3 3 (kỳ vọng 3 )


Sau khi đổi một bước lớn nhất sang thang, `bricks` chắc chắn `≥ 0`: gọi `b₀ ≥ 0` là số gạch **trước** bước này; sau khi trả `d` thì `bricks = b₀ − d < 0`; vì bước vừa thêm cũng nằm trong heap nên `big ≥ d` ⇒ `bricks + big ≥ b₀ ≥ 0`. Mỗi bước leo vào/ra heap nhiều nhất một lần ⇒ `O(n log n)` thời gian, `O(n)` bộ nhớ. Khi `ladders` lớn, cách này dùng ít heap hơn; khi `ladders` nhỏ, cách 4 gọn hơn.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 2: `heights = [4,12,2,7,3,18,20,3,19]`, `bricks = 10`, `ladders = 2`.

| i → i+1 | d | heap sau push | quá L? | pop (trả gạch) | bricks |
|---|---|---|---|---|---|
| 0→1 | 8 | [8] | không | — | 10 |
| 1→2 | −10 | — | — | — | 10 |
| 2→3 | 5 | [5, 8] | không | — | 10 |
| 3→4 | −4 | — | — | — | 10 |
| 4→5 | 15 | [5, 8, 15] | có | 5 | 5 |
| 5→6 | 2 | [2, 8, 15] | có | 2 | 3 |
| 6→7 | −17 | — | — | — | 3 |
| 7→8 | 16 | [8, 15, 16] | có | 8 | **−5** ⇒ trả `7` |

In [13]:
def trace_minheap(heights, bricks, ladders):
    heap = []
    print(f"bricks={bricks}, ladders={ladders}")
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d <= 0:
            print(f"  {i}->{i + 1}: d={d:>3} đi xuống/ngang, miễn phí")
            continue
        heapq.heappush(heap, d)
        msg = f"  {i}->{i + 1}: d={d:>3} push -> heap={sorted(heap)}"
        if len(heap) > ladders:
            small = heapq.heappop(heap)
            bricks -= small
            msg += f" | quá {ladders} thang: trả gạch cho {small}, bricks={bricks}"
            if bricks < 0:
                print(msg + f"  => không sang được {i + 1}, trả về {i}")
                return i
        else:
            msg += " | tạm dùng thang"
        print(msg)
    print(f"  tới tòa cuối {len(heights) - 1}")
    return len(heights) - 1

trace_minheap([4, 12, 2, 7, 3, 18, 20, 3, 19], 10, 2)

bricks=10, ladders=2
  0->1: d=  8 push -> heap=[8] | tạm dùng thang
  1->2: d=-10 đi xuống/ngang, miễn phí
  2->3: d=  5 push -> heap=[5, 8] | tạm dùng thang
  3->4: d= -4 đi xuống/ngang, miễn phí
  4->5: d= 15 push -> heap=[5, 8, 15] | quá 2 thang: trả gạch cho 5, bricks=5
  5->6: d=  2 push -> heap=[2, 8, 15] | quá 2 thang: trả gạch cho 2, bricks=3
  6->7: d=-17 đi xuống/ngang, miễn phí
  7->8: d= 16 push -> heap=[8, 15, 16] | quá 2 thang: trả gạch cho 8, bricks=-5  => không sang được 8, trả về 7


7

Trace cách 5 (max-heap) trên cùng ví dụ để thấy hai góc nhìn cho cùng đáp án:

In [14]:
def trace_maxheap(heights, bricks, ladders):
    heap = []
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d <= 0:
            continue
        bricks -= d
        heapq.heappush(heap, -d)
        msg = f"  {i}->{i + 1}: d={d:>3} trả gạch, bricks={bricks:>3}"
        if bricks < 0:
            if ladders == 0:
                print(msg + f" | hết thang => trả về {i}")
                return i
            big = -heapq.heappop(heap)
            bricks += big
            ladders -= 1
            msg += f" | âm! dùng thang thay cho bước {big}: bricks={bricks}, còn {ladders} thang"
        print(msg)
    return len(heights) - 1

print("Kết quả:", trace_maxheap([4, 12, 2, 7, 3, 18, 20, 3, 19], 10, 2))

  0->1: d=  8 trả gạch, bricks=  2
  2->3: d=  5 trả gạch, bricks= -3 | âm! dùng thang thay cho bước 8: bricks=5, còn 1 thang
  4->5: d= 15 trả gạch, bricks=-10 | âm! dùng thang thay cho bước 15: bricks=5, còn 0 thang
  5->6: d=  2 trả gạch, bricks=  3
  7->8: d= 16 trả gạch, bricks=-13 | hết thang => trả về 7
Kết quả: 7


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def furthestBuilding(self, heights: List[int], bricks: int, ladders: int) -> int:
        heap = []                         # min-heap các bước leo đang "dùng thang"
        for i in range(len(heights) - 1):
            d = heights[i + 1] - heights[i]
            if d > 0:
                heapq.heappush(heap, d)
                if len(heap) > ladders:
                    bricks -= heapq.heappop(heap)
                    if bricks < 0:
                        return i
        return len(heights) - 1

for h, b, l, exp in examples:
    assert Solution().furthestBuilding(h, b, l) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Bổ đề 1 (hoán đổi):** với tập bước leo `D` cố định cần vượt qua, số gạch tối thiểu là `tổng(D) − tổng(L phần tử lớn nhất của D)` (nếu `|D| ≤ L` thì là 0). Chứng minh: nếu phương án có thang ở `x` và gạch ở `y > x`, hoán đổi giảm gạch đi `y − x`; nếu còn thang chưa dùng mà có bước trả bằng gạch, chuyển sang thang chỉ giảm gạch. Lặp tới khi thang nằm đúng ở `L` bước lớn nhất.

**Bổ đề 2 (đơn điệu):** đặt `need(k)` = số gạch tối thiểu để tới tòa `k`. Khi thêm một bước leo, `need` không giảm (bỏ đi một phần tử của tập không thể làm tăng tổng phần "nhỏ" … ngược lại thêm vào chỉ làm nó tăng hoặc giữ nguyên). ⇒ Tới được `k` ⇔ `need(k) ≤ bricks`, và tập các `k` tới được là một đoạn đầu `[0, ans]`.

**Bất biến của min-heap:** sau khi xử lý bước `i → i+1` (chưa trả về), heap chứa đúng `min(L, |D_i|)` phần tử lớn nhất của `D_i` (tập bước leo trong `[0, i+1)`), và `bricks_ban_đầu − bricks_hiện_tại = need(i + 1)`. Chứng minh quy nạp: thêm `d` vào heap; nếu kích thước vượt `L`, phần tử nhỏ nhất của heap chính là phần tử **không** thuộc top-`L` mới (mọi phần tử ngoài heap đều ≤ nó), nên chuyển nó sang gạch giữ đúng bất biến.

Vậy lần đầu `bricks < 0` tại bước `i → i+1` nghĩa là `need(i+1) > bricks_ban_đầu` ⇒ không tới được `i + 1`, và theo bổ đề 2 không tới được tòa nào xa hơn; trong khi `need(i) ≤ bricks` ⇒ đáp án là `i`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(heights)`, `L = ladders`, `c` = số bước leo (`c ≤ n − 1`).

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^c)` | `O(n)` stack |
| 2. DP theo số thang | `O(n · L)` | `O(L)` |
| 3. Tìm kiếm nhị phân | `O(n log² n)` | `O(n)` |
| 4. Min-heap cỡ `L` | `O(n log L)` | `O(L)` |
| 5. Max-heap | `O(n log n)` | `O(n)` |

Đo thời gian trên input cỡ tối đa (`n = 10⁵`):

In [16]:
rnd = random.Random(1642)
H = [rnd.randint(1, 10**6) for _ in range(10**5)]
B, L = 10**9 // 50, 2000
for name, fn in [("binary", furthest_binary), ("minheap", furthest_minheap), ("maxheap", furthest_maxheap)]:
    t0 = time.perf_counter(); r = fn(H, B, L); dt = time.perf_counter() - t0
    print(f"{name:<8}: {r:>6}  {dt * 1000:8.1f} ms")
t0 = time.perf_counter(); r = furthest_dp(H[:3000], B, 300); dt = time.perf_counter() - t0
print(f"dp (n=3000, L=300): {r} {dt * 1000:.1f} ms  | minheap cùng input: {furthest_minheap(H[:3000], B, 300)}")

binary  :   4874      18.4 ms
minheap :   4874       0.5 ms
maxheap :   4874       0.7 ms
dp (n=3000, L=300): 1001 10.9 ms  | minheap cùng input: 1001


## 12. Bộ test

Đối chiếu: ví dụ đề; edge case (`n = 1`, toàn đi xuống, không gạch không thang, đủ thang cho mọi bước, đúng bằng số gạch); và ngẫu nhiên nhỏ có seed cố định so với brute force (đúng ràng buộc: `heights[i] ≥ 1`, `ladders ≤ n`).

In [17]:
ALL = {
    "dp": furthest_dp,
    "binary": furthest_binary,
    "minheap": furthest_minheap,
    "maxheap": furthest_maxheap,
    "Solution": lambda h, b, l: Solution().furthestBuilding(h, b, l),
    "SolBinary": lambda h, b, l: SolutionBinary().furthestBuilding(h, b, l),
    "SolMaxHeap": lambda h, b, l: SolutionMaxHeap().furthestBuilding(h, b, l),
}

def check(h, b, l, exp=None, brute=True):
    ref = furthest_brute(h, b, l) if brute else ALL["dp"](h, b, l)
    for name, fn in ALL.items():
        got = fn(list(h), b, l)
        assert got == ref, (name, h, b, l, got, ref)
    if exp is not None:
        assert ref == exp, (h, b, l, ref, exp)
    return ref

for h, b, l, exp in examples:
    check(h, b, l, exp)
print("Ví dụ đề: OK")

check([7], 0, 0, 0)                          # một tòa
check([5, 4, 3, 2, 1], 0, 0, 4)              # toàn đi xuống
check([1, 2], 0, 0, 0)                       # không tài nguyên
check([1, 5, 9, 20], 0, 3, 3)                # đủ thang
check([1, 5, 9], 8, 0, 2)                    # vừa khít số gạch (4 + 4)
check([1, 5, 9], 7, 0, 1)                    # thiếu 1 viên
check([1, 1, 1, 1], 0, 0, 3)                 # đi ngang
check([3, 19, 3, 19], 0, 1, 2)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [18]:
rnd = random.Random(2023)
for _ in range(3000):
    n = rnd.randint(1, 11)
    h = [rnd.randint(1, 12) for _ in range(n)]
    b = rnd.randint(0, 20)
    l = rnd.randint(0, n)
    check(h, b, l)
print("3000 bộ ngẫu nhiên nhỏ so với brute force: OK")

for _ in range(40):
    n = rnd.randint(50, 400)
    h = [rnd.randint(1, 10**6) for _ in range(n)]
    b = rnd.randint(0, 10**7)
    l = rnd.randint(0, min(n, 30))
    check(h, b, l, brute=False)
print("40 bộ cỡ vừa so với DP: OK")

3000 bộ ngẫu nhiên nhỏ so với brute force: OK


40 bộ cỡ vừa so với DP: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam cục bộ "còn gạch thì dùng gạch, hết mới dùng thang"

In [19]:
def furthest_bug_bricks_first(heights, bricks, ladders):
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d <= 0:
            continue
        if bricks >= d:
            bricks -= d
        elif ladders > 0:
            ladders -= 1
        else:
            return i
    return len(heights) - 1

h, b, l = [1, 6, 7, 8], 5, 1
print("sai :", furthest_bug_bricks_first(h, b, l), "| đúng:", Solution().furthestBuilding(h, b, l), "| brute:", furthest_brute(h, b, l))
print("các bước leo:", climbs(h))

sai : 2 | đúng: 3 | brute: 3
các bước leo: [(0, 5), (1, 1), (2, 1)]


Ở ví dụ `[1, 6, 7, 8]`, 5 gạch, 1 thang: tham lam cục bộ trả 5 gạch cho bước leo `5`, dùng thang cho bước `1`, rồi bước `1` cuối hết tài nguyên ⇒ dừng ở tòa `2`. Phương án tối ưu: **thang cho bước `5`**, gạch cho hai bước `1` ⇒ tới tòa `3`.

Một thói quen tốt: **đừng chỉ tin phản ví dụ tự nghĩ — hãy để brute force tìm giúp**. Ô sau sinh ngẫu nhiên (seed cố định) cho tới khi gặp input mà tham lam cục bộ khác brute force:

In [20]:
rnd = random.Random(5)
found = None
while found is None:
    n = rnd.randint(2, 6)
    h = [rnd.randint(1, 10) for _ in range(n)]
    b, l = rnd.randint(0, 8), rnd.randint(0, 2)
    if furthest_bug_bricks_first(h, b, l) != furthest_brute(h, b, l):
        found = (h, b, l)
h, b, l = found
print("Phản ví dụ tìm được:", found)
print("   tham lam cục bộ:", furthest_bug_bricks_first(h, b, l), "| đúng:", furthest_brute(h, b, l), "| Solution:", Solution().furthestBuilding(h, b, l))
print("   các bước leo:", climbs(h))

Phản ví dụ tìm được: ([7, 4, 7, 8, 9], 3, 1)
   tham lam cục bộ: 3 | đúng: 4 | Solution: 4
   các bước leo: [(1, 3), (2, 1), (3, 1)]


**Vì sao sai (tổng quát):** tham lam cục bộ dùng gạch cho bước **lớn** khi còn gạch, rồi về sau không còn gạch cho các bước **nhỏ** — trong khi lẽ ra thang phải dành cho bước lớn. **Sửa:** quyết định "hồi tố" bằng heap: thang luôn thuộc về `L` bước lớn nhất đã gặp.

### Lỗi 2 — Trả về `i + 1` (hoặc `i − 1`) khi thiếu gạch

In [21]:
def furthest_bug_offby1(heights, bricks, ladders):
    heap = []
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d > 0:
            heapq.heappush(heap, d)
            if len(heap) > ladders:
                bricks -= heapq.heappop(heap)
                if bricks < 0:
                    return i + 1          # SAI: chưa sang được i + 1
    return len(heights) - 1

h, b, l, exp = examples[0]
print("sai :", furthest_bug_offby1(h, b, l), "| đúng:", Solution().furthestBuilding(h, b, l))

sai : 5 | đúng: 4


**Vì sao sai:** đang xét bước `i → i+1`; nếu thiếu gạch thì ta **đứng ở `i`**. **Sửa:** `return i`.

### Lỗi 3 — Đẩy cả bước đi xuống (d ≤ 0) vào heap

In [22]:
def furthest_bug_push_all(heights, bricks, ladders):
    heap = []
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        heapq.heappush(heap, d)                 # SAI: đẩy cả d âm
        if len(heap) > ladders:
            bricks -= heapq.heappop(heap)
            if bricks < 0:
                return i
    return len(heights) - 1

h, b, l = [1, 2, 1, 2, 1, 9], 1, 1
print("sai :", furthest_bug_push_all(h, b, l), "| đúng:", Solution().furthestBuilding(h, b, l))

sai : 5 | đúng: 4


**Vì sao sai:** bước đi xuống chiếm một "suất thang" trong heap, và khi bị pop, `bricks −= d` với `d` âm lại **tăng** gạch một cách vô lý. **Sửa:** chỉ xử lý khi `d > 0`.

### Lỗi 4 — Kiểm tra `bricks < 0` ở ngoài nhánh pop với max-heap nhưng quên `ladders == 0`

Trong cách 5, nếu khi `bricks < 0` ta luôn pop và `ladders −= 1` mà không kiểm tra còn thang, `ladders` thành âm và kết quả sai:

In [23]:
def furthest_bug_maxheap(heights, bricks, ladders):
    heap = []
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d > 0:
            bricks -= d
            heapq.heappush(heap, -d)
            if bricks < 0:
                bricks -= heapq.heappop(heap)     # SAI: không kiểm tra ladders
                ladders -= 1
    return len(heights) - 1

h, b, l = [14, 3, 19, 30], 17, 0
print("sai :", furthest_bug_maxheap(h, b, l), "| đúng:", Solution().furthestBuilding(h, b, l))

sai : 3 | đúng: 2


**Sửa:** `if ladders == 0: return i` trước khi dùng thang.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Có hai loại tài nguyên: một loại **trả theo kích thước** (gạch), một loại **trả theo số lần** (thang)" ⇒ loại theo số lần nên dành cho **các mục lớn nhất** ⇒ **top-k bằng heap**.
- "Quyết định có thể **sửa lại về sau**" (regret greedy) ⇒ giữ các quyết định trong heap để đổi khi cần.
- Đáp án là "đi xa nhất" và tính khả thi **đơn điệu** ⇒ luôn có phương án dự phòng: **tìm kiếm nhị phân đáp án**.
- Min-heap cỡ `k` để giữ **k phần tử lớn nhất**; max-heap (đổi dấu) để giữ phần tử lớn nhất cần "hoàn tác".
- Cẩn thận chỉ số trả về: đang ở bước `i → i+1` thì thất bại nghĩa là đáp án `i`.

## 15. Biến thể của bài

1. **Mỗi thang có độ dài giới hạn `len`:** thang chỉ dùng cho bước `d ≤ len` — heap vẫn dùng được nhưng phải lọc các bước quá cao (bắt buộc gạch).
2. **Hai loại thang với độ dài khác nhau:** quy về gán top-k theo hai tầng.
3. **Có thể đi cả hai chiều / chọn thứ tự tòa nhà:** thành bài đồ thị (Dijkstra theo tài nguyên).
4. **LeetCode 871 — Minimum Number of Refueling Stops:** cùng kiểu "regret greedy" với max-heap: đi tới đâu, nếu thiếu xăng thì "hồi tố" đổ xăng ở trạm lớn nhất đã đi qua.
5. **LeetCode 630 — Course Schedule III:** chọn khóa học, khi vượt deadline thì bỏ khóa dài nhất (max-heap).
6. **Đếm số gạch tối thiểu để tới tòa cuối** (luôn tới được nếu có đủ gạch): `tổng − tổng top-L` — một lần sắp xếp.

In [24]:
def min_bricks_to_end(heights, ladders):
    D = sorted((b - a for a, b in zip(heights, heights[1:]) if b > a), reverse=True)
    return sum(D[ladders:])

h = [4, 12, 2, 7, 3, 18, 20, 3, 19]
for L in range(0, 6):
    need = min_bricks_to_end(h, L)
    print(f"ladders={L}: cần {need} gạch để tới tòa cuối | kiểm tra Solution:", Solution().furthestBuilding(h, need, L) == len(h) - 1,
          "| bớt 1 viên:", Solution().furthestBuilding(h, need - 1, L) if need > 0 else "-")

ladders=0: cần 46 gạch để tới tòa cuối | kiểm tra Solution: True | bớt 1 viên: 7
ladders=1: cần 30 gạch để tới tòa cuối | kiểm tra Solution: True | bớt 1 viên: 7
ladders=2: cần 15 gạch để tới tòa cuối | kiểm tra Solution: True | bớt 1 viên: 7
ladders=3: cần 7 gạch để tới tòa cuối | kiểm tra Solution: True | bớt 1 viên: 7
ladders=4: cần 2 gạch để tới tòa cuối | kiểm tra Solution: True | bớt 1 viên: 7
ladders=5: cần 0 gạch để tới tòa cuối | kiểm tra Solution: True | bớt 1 viên: -


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử gạch/thang mỗi bước leo | `O(2^c)` | `O(n)` | đối chiếu với input rất nhỏ |
| 2. DP | `dp[j]` = gạch ít nhất khi dùng `j` thang | `O(n·L)` | `O(L)` | khi không thấy tham lam; L nhỏ |
| 3. Nhị phân đáp án | `ok(k)` = top-L của `k` bước đầu | `O(n log² n)` | `O(n)` | an toàn, dễ chứng minh |
| 4. Min-heap cỡ L | giữ L bước cao nhất trên đường | `O(n log L)` | `O(L)` | **nộp bài** |
| 5. Max-heap | gạch trước, âm thì đổi bước lớn nhất sang thang | `O(n log n)` | `O(n)` | khuôn "regret greedy" tổng quát |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().furthestBuilding([1, 13, 1, 13, 1], 11, 1)` bằng bao nhiêu?

**Bài 2 (tính tay):** Trace cách 4 cho `heights = [2, 7, 3, 6, 10, 4, 8]`, `bricks = 4`, `ladders = 2`: ghi heap và `bricks` sau mỗi bước leo, cho biết đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `which_use_ladders(heights, bricks, ladders)` trả về **danh sách chỉ số bước** `i` (bước `i → i+1`) được dùng thang trong phương án tối ưu khi đi tới tòa xa nhất.

**Bài 4 (tự viết lại):** Viết lại cách 3 (tìm kiếm nhị phân) nhưng dùng `heapq.nlargest` thay vì sắp xếp toàn bộ.

**Bài 5 (mini challenge):** Biến thể "thang có độ dài tối đa `maxlen`": thang chỉ dùng được cho bước `d ≤ maxlen`. Viết `furthest_limited_ladder(heights, bricks, ladders, maxlen)` và kiểm bằng brute force.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def which_use_ladders(heights, bricks, ladders):
    # TODO: chạy min-heap nhưng lưu (d, i); khi dừng, các phần tử còn trong heap là các bước dùng thang
    pass

# Bài 4 — TODO
def furthest_binary_nlargest(heights, bricks, ladders):
    # TODO: ok(k) = sum(D) - sum(heapq.nlargest(ladders, D)) <= bricks
    pass

# Bài 5 — TODO
def furthest_limited_ladder(heights, bricks, ladders, maxlen):
    # TODO: bước d > maxlen bắt buộc dùng gạch; các bước còn lại xử lý như min-heap
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Có hai bước leo, đều bằng 12.
- Mức 2: Một bước dùng thang, bước kia cần 12 gạch.
- Mức 3: Chỉ có 11 gạch.

**Bài 2**
- Mức 1: Các bước leo: `5, 3, 4, 4`.
- Mức 2: Heap tối đa 2 phần tử; phần tử thứ 3 làm pop phần tử nhỏ nhất.
- Mức 3: Sau bước `3` (pop 3) còn 1 gạch; bước `4` cuối pop 4 ⇒ âm.

**Bài 3**
- Mức 1: Lưu tuple `(d, i)` trong heap.
- Mức 2: Khi trả về sớm, phần tử vừa bị pop **không** được dùng gạch thành công — các phần tử trong heap trước lần pop đó là thang.
- Mức 3: Chỉ cần lấy các chỉ số còn trong heap sau khi đã xác định tòa cuối cùng tới được (chạy lại tới đáp án).

**Bài 4**
- Mức 1: `nlargest(k, D)` trả `k` phần tử lớn nhất trong `O(len(D) log k)`.
- Mức 2: Cẩn thận `ladders = 0`: `nlargest(0, D) = []`.
- Mức 3: Phần tìm kiếm nhị phân giữ nguyên.

**Bài 5**
- Mức 1: Bước quá cao trừ thẳng vào `bricks`.
- Mức 2: Bước thấp hơn hoặc bằng `maxlen` đi qua min-heap như cũ.
- Mức 3: Kiểm `bricks < 0` sau **cả hai** loại trừ gạch.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
ans1 = Solution().furthestBuilding([1, 13, 1, 13, 1], 11, 1)
print("Bài 1:", ans1)
assert ans1 == 2 == furthest_brute([1, 13, 1, 13, 1], 11, 1)

# Bài 2
ans2 = trace_minheap([2, 7, 3, 6, 10, 4, 8], 4, 2)
assert ans2 == furthest_brute([2, 7, 3, 6, 10, 4, 8], 4, 2)

Bài 1: 2
bricks=4, ladders=2
  0->1: d=  5 push -> heap=[5] | tạm dùng thang
  1->2: d= -4 đi xuống/ngang, miễn phí
  2->3: d=  3 push -> heap=[3, 5] | tạm dùng thang
  3->4: d=  4 push -> heap=[3, 4, 5] | quá 2 thang: trả gạch cho 3, bricks=1
  4->5: d= -6 đi xuống/ngang, miễn phí
  5->6: d=  4 push -> heap=[4, 4, 5] | quá 2 thang: trả gạch cho 4, bricks=-3  => không sang được 6, trả về 5


In [27]:
# Bài 3
def which_use_ladders_sol(heights, bricks, ladders):
    target = Solution().furthestBuilding(heights, bricks, ladders)
    heap = []
    for i in range(target):                       # chỉ chạy tới tòa đích, chắc chắn không âm
        d = heights[i + 1] - heights[i]
        if d > 0:
            heapq.heappush(heap, (d, i))
            if len(heap) > ladders:
                heapq.heappop(heap)
    return sorted(i for _, i in heap)

def plan_ok(heights, bricks, ladder_steps, target):
    used = 0
    for i in range(target):
        d = heights[i + 1] - heights[i]
        if d > 0 and i not in ladder_steps:
            used += d
    return used <= bricks

rnd = random.Random(31)
for _ in range(1500):
    n = rnd.randint(1, 10)
    h = [rnd.randint(1, 12) for _ in range(n)]
    b, l = rnd.randint(0, 15), rnd.randint(0, n)
    steps = which_use_ladders_sol(h, b, l)
    assert len(steps) <= l and plan_ok(h, b, set(steps), Solution().furthestBuilding(h, b, l))
print("Bài 3 OK | ví dụ 2:", which_use_ladders_sol([4, 12, 2, 7, 3, 18, 20, 3, 19], 10, 2))

# Bài 4
def furthest_binary_nlargest_sol(heights, bricks, ladders):
    def ok(k):
        D = [heights[i + 1] - heights[i] for i in range(k) if heights[i + 1] > heights[i]]
        return sum(D) - sum(heapq.nlargest(ladders, D)) <= bricks
    lo, hi = 0, len(heights) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        lo, hi = (mid, hi) if ok(mid) else (lo, mid - 1)
    return lo

for _ in range(1500):
    n = rnd.randint(1, 12)
    h = [rnd.randint(1, 15) for _ in range(n)]
    b, l = rnd.randint(0, 20), rnd.randint(0, n)
    assert furthest_binary_nlargest_sol(h, b, l) == Solution().furthestBuilding(h, b, l)
print("Bài 4 OK")

Bài 3 OK | ví dụ 2: [0, 4]
Bài 4 OK


In [28]:
# Bài 5
def furthest_limited_ladder_sol(heights, bricks, ladders, maxlen):
    heap = []
    for i in range(len(heights) - 1):
        d = heights[i + 1] - heights[i]
        if d <= 0:
            continue
        if d > maxlen:
            bricks -= d                       # bắt buộc dùng gạch
        else:
            heapq.heappush(heap, d)
            if len(heap) > ladders:
                bricks -= heapq.heappop(heap)
        if bricks < 0:
            return i
    return len(heights) - 1

def furthest_limited_brute(heights, bricks, ladders, maxlen):
    n = len(heights)
    def go(i, b, l):
        if i == n - 1:
            return i
        d = heights[i + 1] - heights[i]
        if d <= 0:
            return go(i + 1, b, l)
        best = i
        if b >= d:
            best = max(best, go(i + 1, b - d, l))
        if l > 0 and d <= maxlen:
            best = max(best, go(i + 1, b, l - 1))
        return best
    return go(0, bricks, ladders)

rnd = random.Random(55)
for _ in range(2500):
    n = rnd.randint(1, 10)
    h = [rnd.randint(1, 12) for _ in range(n)]
    b, l, m = rnd.randint(0, 15), rnd.randint(0, n), rnd.randint(0, 12)
    assert furthest_limited_ladder_sol(h, b, l, m) == furthest_limited_brute(h, b, l, m), (h, b, l, m)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "giữ top-k phần tử lớn nhất trên đường đi" (min-heap cỡ k):**

```python
heap = []
for i, cost in enumerate(costs):        # chỉ các cost > 0
    heapq.heappush(heap, cost)
    if len(heap) > k:
        budget -= heapq.heappop(heap)   # phần tử nhỏ nhất trả bằng ngân sách
        if budget < 0:
            return i                    # thất bại ở bước i
return len(costs)                       # (tùy bài: chỉ số cuối)
```

**Template "regret greedy" (max-heap):** trả bằng ngân sách trước, `push(-cost)`; khi âm thì `budget += -heappop(heap)` và tiêu một "vé hoàn tác".

**Checklist:**
- [ ] Chỉ xử lý bước **leo lên** (`d > 0`)?
- [ ] Thang dành cho các bước **cao nhất** (min-heap cỡ `ladders`)?
- [ ] Trả về `i` (không phải `i + 1`) khi thiếu gạch ở bước `i → i+1`?
- [ ] Khi đi hết, trả `len(heights) − 1`?
- [ ] Trường hợp `ladders = 0` và `n = 1` chạy đúng?
- [ ] Đã so với brute force trên vài nghìn input ngẫu nhiên nhỏ?
- [ ] Chữ ký đúng: `def furthestBuilding(self, heights: List[int], bricks: int, ladders: int) -> int`?